In [ ]:
import pandas as pd
import json

In [ ]:
conversations_df = pd.read_csv("CONVERSATIONS.csv")
conversation_parts_df = pd.read_csv("CONVERSATION_PARTS.csv")

clean_conversations = conversations_df[[
    'ID', 'ASSIGNEE', 'CREATED_AT', 'UPDATED_AT', 'CONVERSATION_RATING', 'WAITING_SINCE'
]].copy()
clean_conversations.columns = ['conversation_id', 'assignee_id', 'created_at', 'updated_at', 'conversation_rating', 'waiting_since']

<ipython-input-3-55cca24a661c>:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  conversation_parts_df = pd.read_csv("CONVERSATION_PARTS.csv")


In [ ]:
# Nettoyage des messages
def parse_author_id(author):
    try:
        return json.loads(author.replace('\n', '').replace("'", '"')).get('id')
    except:
        return None

def parse_author_type(author):
    try:
        return json.loads(author.replace('\n', '').replace("'", '"')).get('type')
    except:
        return None

filtered_parts = conversation_parts_df.copy()
filtered_parts['author_id'] = filtered_parts['AUTHOR'].apply(parse_author_id)
filtered_parts['author_type'] = filtered_parts['AUTHOR'].apply(parse_author_type)

In [ ]:
# Filtrage : messages humains seulement
filtered_parts = filtered_parts[
    (filtered_parts['PART_GROUP'] == 'Message') &
    (filtered_parts['author_type'] != 'bot')
]

conversation_messages = filtered_parts[[
    'CONVERSATION_ID', 'CREATED_AT', 'author_id', 'author_type', 'PART_GROUP'
]].copy()
conversation_messages.columns = ['conversation_id', 'created_at', 'author_id', 'author_type', 'part_group']

In [ ]:
# Limiter à 50 lignes pour exemple ---
clean_conversations_sample = clean_conversations.head(50)
conversation_messages_sample = conversation_messages.head(50)

In [ ]:
# Génération du fichier SQL
def generate_insert_sql(table_name, df):
    insert_statements = []
    for _, row in df.iterrows():
        values = []
        for value in row:
            if pd.isna(value):
                values.append("NULL")
            else:
                escaped = str(value).replace("'", "''")
                values.append(f"'{escaped}'")
        insert_statements.append(f"INSERT INTO {table_name} VALUES ({', '.join(values)});")
    return '\n'.join(insert_statements)

sql_conversations = generate_insert_sql("conversations", clean_conversations_sample)
sql_parts = generate_insert_sql("conversation_messages", conversation_messages_sample)

full_sql_script = f"""
-- Création des tables

CREATE TABLE conversations (
    conversation_id VARCHAR(50),
    assignee_id VARCHAR(50),
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    conversation_rating VARCHAR(10),
    waiting_since TIMESTAMP
);

CREATE TABLE conversation_messages (
    conversation_id VARCHAR(50),
    created_at TIMESTAMP,
    author_id VARCHAR(50),
    author_type VARCHAR(20),
    part_group VARCHAR(20)
);

-- Insertion des données (50 premières lignes)

-- Table: conversations
{sql_conversations}

-- Table: conversation_messages
{sql_parts}
"""

In [ ]:
# Sauvegarde du fichier SQL ---
with open("skello_case_study_final_fixed.sql", "w", encoding="utf-8") as f:
    f.write(full_sql_script)

# Télécharger le fichier SQL ---
files.download("skello_case_study_final_fixed.sql")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>